# Datamine COMRES Process Example

This notebook demonstrates how to configure and run the **`comres`** process wrapper in `dmstudio`.

## Process Description

## COMRES

Generates a file that contains summary results from a number of different **RESULTS** files, that have been produced by processes such as [MODRES](<modres.md>) or [TRIVAL](<trival.md>).

The output file generated is suitable for subsequent mine planning and scheduling purposes, such as interactive production scheduling (**PRODSH**). The file will contain reserve figures for production units that have been classified by the user into primary and secondary categories, and it will contain all grade fields that are contained in the input **RESULTS** files.

Production scheduling requires some initial definition of the quantities and grades of the mining units to be scheduled. The final product of any evaluation is a comprehensive **RESULTS** file that contains data describing the mining reserves. There are a number of standard fields in a **RESULTS** file, which allow each logical part of an evaluation to be stored as a data record with entries including the tonnage, volume, grade(s), grade intervals, block model position, perimeter numbers and wireframe zone numbers.

The **COMRES** process accepts the input from one or more such **RESULTS** files (maximum 20) and generates a **RESERVE** file, which will be used for subsequent production scheduling. Each record of the **RESERVE** file describes the tonnage and grade of a production unit. The classification of production units stems from user selection of various options during the operation of **COMRES**. This allows production units to be classified into primary and secondary categories, based on criteria available from the **RESULTS** files. This flexibility allows the scheduling extension module to be used for both open pit and underground mining applications.

Two numeric fields in the **RESERVE** file, **PNUM** and **SNUM** , contain the primary and secondary numbers for each production unit. The data entries in these fields come directly from the input **RESULTS** files or are assigned logically according to the selected classification option, e.g. if the secondary classification is based on perimeters, the **SNUM** values will come from the **PERIMID** values in the **RESULTS** file(s). If a number of **RESULTS** files are input, and primary classification by model is required, the **PNUM** entries will increment 1, 2, 3 etc. for each **RESULTS** file.

If secondary classification is by ore/waste, the **SNUM** values will be either 1 for ore or 2 for waste. Another alphanumeric field, **UNIT** , allows a clear primary descriptor to be assigned for each production unit, which will appear on the schedule screen in the **PRODSH** process.

The production rate at which each production unit will be mined may also be defined in the **COMRES** process. The **RESERVE** file is a normal Datamine binary file, and so may be listed, edited or joined with other files using the available relational database facilities.

### Results Files

Enter the name of each **RESULTS** file, followed by <return>. Typing <?> will show a list of the currently selected **RESULTS** files. Typing a back arrow, <, will clear all the filenames entered and reprompt from the beginning. Entries are terminated by just typing <return>. Pressing <!> will terminate the process. When more than one **RESULTS** file is entered, it is expected that each contains the same grade fields.

### Production Rates

Enter required production rate for each **RESULTS** file. Pressing <return> will put absent data into the production rate data record for that **RESULTS** file. These production rate values may be initialized or modified in the **PRODSH** production scheduling process. These values will be stored in the **PRATE** field of the **RESERVE** file.

### Production Unit Classification

For production scheduling purpose, production units are classified from data held in the **RESULTS** files, and this may be done in a number of ways. A primary and optionally a secondary means of classification is available, allowing a number of features during production scheduling.

If more than one **RESULTS** file has been entered, the user will get the following prompt:

## > IS THE SAME TYPE OF PRIMARY AND SECONDARY CLASSIFICATION REQUIRED

## FOR EACH RESULTS FILE [Y]/N  >

If N or NO is entered, you will get primary and secondary classification options for each **RESULTS** file that has been entered.

If the default option has been taken for the above prompt, then the classification selected below will used for each of the supplied **RESULTS** files.

### Primary Classification

Options are:

1. Model or Sample File
2. Plane
3. Perimeter
4. Zone

Enter number for required type of primary classification, which will determine the manner of primary tonnage and grade accumulation, in addition to the primary names and numbers in the output **RESERVE** file. Just typing <return> will take the default 'MODEL' or 'SAMPLE FILE' (from a **SECTED** evaluation) primary classification.

### Secondary Classification

Options are:

1. No secondary classification
2. Plane
3. Perimeter
4. Zone
5. Ore/Waste
6. Grade Interval

Enter number for required type of secondary classification. Just typing <return> will take the default, omitting any secondary classification.

These primary and secondary classification prompts will be repeated for each **RESULTS** file entered, if a a separate means of classification for each file was requested above.

Once the primary and secondary classification is complete the **RESERVE** file will be generated.

### Input Files:

### Output Files:

* **reserve** (Undefined):
  The output file generated, for use in subsequent scheduling processes, such as
  **PRODSH**. It will contain the following fields, in addition to all of the grade fields
  in the **RESULTS** file(s). **UNIT** A,8 Field containing the name of each production
  unit. **PNUM** N Any blocks being scheduled may be categorised according to a primary
  and secondary classification. The **PNUM** field contains the primary number. The values
  held in this field will depend on the type of primary classification selected. **SNUM**
  N Secondary classification number. The values held in this field will depend on the type
  of secondary classification selected. If ore/waste secondary classification is selected,
  it will contain values of 1 for all ore units, and values of 2 for all waste units.
  **NY** N Implicit field, defining the total number of production units in the file,
  equal to the number of records. **PRATE** N Notional production rate at which each
  production unit is to be mined. This may contain absent data values, as individual
  production rates may be defined or changed during operation of **PRODSH**. **TONNES** N
  Reserve tonnage for each production unit.
  Required=Yes

### Fields:

* **zone** (Numeric : RESERVE):
  Optional numeric zone identifier field that has been used to define individual
  wireframe zones. This field need only be defined if reserve classification by wireframe
  zone is required.
  Default=Undefined
  Required=No

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('comres')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute comres
print("Running comres...")
dm_fil.comres(
    reserve_o='t_comres_out',  # required
    # zone_f='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("comres execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_comres_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")